# Synthetic Test: `popexposure`

This notebook builds a fully synthetic test case for `popexposure` with known population
values, so we can verify the results analytically.

## Setup

- **Population raster**: 100 × 100 grid of 1 km × 1 km cells starting at null island (0°, 0°) in WGS84.
  - Bottom half (y = 0–50 km, rows 50–99): **1 person per cell**
  - Top half (y = 50–100 km, rows 0–49): **2 persons per cell**
- **Hazards** (coordinates in km from origin):
  1. `point_near`: Point at (5 km, 5 km) — bottom half
  2. `point_far`: Point at (55 km, 55 km) — top half
  3. `line`: LineString from (10 km, 10 km) to (10 km, 20 km) — bottom half
  4. `polygon`: Small square centred at (60 km, 60 km) ± 2 km — top half
  5. `missing_geom`: Row with a `None` geometry
- **Buffer distances**: 1 km (1000 m) and 2 km (2000 m) for all hazards

In [ ]:
# Additional library imports for creating synthetic test case and executing
import numpy as np
import rasterio
from rasterio.transform import from_origin
from rasterio.crs import CRS
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point, LineString, Polygon
import tempfile, pathlib, warnings
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# PopExposure!
import popexposure

In [ ]:
# Make the raster for the synthetic test case
# Going to make 1km x 1km raster starting at 0, 0 degrees
# Population raster: 100 × 100 grid of 1 km × 1 km cells starting at null island (0°, 0°) in WGS84.
# Bottom half (y = 0–50 km, rows 50–99):1 person per cell
# Top half (y = 50–100 km, rows 0–49):2 persons per cell


KM_PER_DEG = 111.32        # km per degree (equatorial approximation)
CELL_KM    = 1.0           # 1 km x 1 km cells
N_CELLS    = 100           # 100 x 100 grid -> 100 km x 100 km total

cell_deg = CELL_KM / KM_PER_DEG   # ~ 0.008983 deg per cell

# Rasterio origin is top left corner
origin_lon = 0.0
origin_lat = N_CELLS * cell_deg   # ~ 0.8983 deg N (top of raster)

# Define affine transform
transform = from_origin(
    west  = origin_lon, # origin lon
    north = origin_lat, # origin lat
    xsize = cell_deg, # how big is x dim of cells and in what direction
    ysize = cell_deg,  # how big is y dim and in what direction
)

# Create the actual array with values
pop_array = np.empty((N_CELLS, N_CELLS), dtype=np.float32)
pop_array[:50, :] = 2.0
pop_array[50:, :] = 1.0

In [ ]:
# Write to GeoTIFF
raster_path = pathlib.Path('synthetic_pop.tif')

with rasterio.open(
    raster_path, mode='w', driver='GTiff',
    height=N_CELLS, width=N_CELLS, count=1,
    dtype=np.float32, crs=CRS.from_epsg(4326),
    transform=transform, nodata=-9999,
) as dst:
    dst.write(pop_array, 1)

In [ ]:
# Plot raster and verify the bounds are right
from rasterio.plot import show

with rasterio.open(raster_path) as src:
    show(src, cmap='YlOrRd')
    print(src.bounds)


In [ ]:
# Make hazard dataframe
# Hazards** (coordinates in km from origin):
# 1. point_near: Point at (5 km, 5 km) — bottom half
# 2. point_far: Point at (55 km, 55 km) — top half
# 3. line: LineString from (10 km, 10 km) to (10 km, 20 km) — bottom half
# 4. polygon: Small square centred at (60 km, 60 km) ± 2 km — top half
# 5. missing_geom: Row with a `None` geometry

geom_point_near = Point(30 / KM_PER_DEG, 30 / KM_PER_DEG)
geom_point_far  = Point(70 / KM_PER_DEG, 70 / KM_PER_DEG)
geom_line       = LineString([(10 / KM_PER_DEG, 10 / KM_PER_DEG),
                               (10 / KM_PER_DEG, 20 / KM_PER_DEG)])
geom_polygon    = Polygon([(58 / KM_PER_DEG, 58 / KM_PER_DEG),
                            (62 / KM_PER_DEG, 58 / KM_PER_DEG),
                            (62 / KM_PER_DEG, 62 / KM_PER_DEG),
                            (58 / KM_PER_DEG, 62 / KM_PER_DEG)])
geom_missing    = None

hazards = gpd.GeoDataFrame(
    {
        'ID_hazard':      ['point_near', 'point_far', 'line', 'polygon', 'missing_geom'],
        'buffer_dist_1': [2000] * 5,
        'buffer_dist_2': [5000] * 5,
        'geometry':       [geom_point_near, geom_point_far,
                           geom_line, geom_polygon, geom_missing],
    },
    crs='EPSG:4326',
)
print(hazards[['ID_hazard', 'buffer_dist_1', 'buffer_dist_2', 'geometry']])

hazards_path = pathlib.Path('synthetic_hazards.geojson')
hazards.to_file(hazards_path, driver='GeoJSON')
hazards.plot()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))

# Plot raster
extent = [0, N_CELLS * cell_deg, 0, N_CELLS * cell_deg]
cmap_grey = mcolors.ListedColormap(['#d0d0d0', '#606060'])
im = ax.imshow(pop_array, origin='upper', extent=extent, cmap=cmap_grey, vmin=0.5, vmax=2.5, alpha=0.65)
cbar = plt.colorbar(im, ax=ax, shrink=0.6)
cbar.set_ticks([1, 2])
cbar.set_ticklabels(['1 person/cell', '2 persons/cell'])

# Viridis purples for hazards
viridis = plt.cm.viridis
hazard_colors = {
    'point_near': mcolors.to_hex(viridis(0.10)),
    'point_far':  mcolors.to_hex(viridis(0.20)),
    'line':       mcolors.to_hex(viridis(0.30)),
    'polygon':    mcolors.to_hex(viridis(0.40)),
}

# Plot hazards and their buffers
for _, row in hazards.dropna(subset=['geometry']).iterrows():
    g, hid = row.geometry, row.ID_hazard
    col = hazard_colors[hid]

    # Buffer in degrees (convert from metres)
    buf_1 = g.buffer(row.buffer_dist_1 / 111320)
    buf_2 = g.buffer(row.buffer_dist_2 / 111320)

    # Draw buffers
    ax.fill(*buf_2.exterior.xy, alpha=0.1, color=col, zorder=3)
    ax.plot(*buf_2.exterior.xy, color=col, lw=0.8, ls='--', zorder=3)
    ax.fill(*buf_1.exterior.xy, alpha=0.15, color=col, zorder=4)
    ax.plot(*buf_1.exterior.xy, color=col, lw=0.8, zorder=4)

    # Draw hazard geometry
    if g.geom_type == 'Point':
        ax.plot(g.x, g.y, 'o', ms=1, color=col, label=hid, zorder=5)
    elif g.geom_type == 'LineString':
        ax.plot(*g.xy, lw=3, color=col, label=hid, zorder=5)
    elif g.geom_type == 'Polygon':
        x, y = g.exterior.xy
        ax.fill(x, y, alpha=0.5, color=col, label=hid, zorder=5)

# 50 km boundary
ax.axhline(50 * cell_deg, color='black', ls='--', lw=1.5, label='1pp/2pp boundary')

ax.set_xlabel('Longitude (deg)')
ax.set_ylabel('Latitude (deg)')
ax.set_title('Hazards and hazard buffer zones from test case\n overlayed on test raster ')
ax.legend(loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))

# Plot raster
extent = [0, N_CELLS * cell_deg, 0, N_CELLS * cell_deg]
cmap_grey = mcolors.ListedColormap(['#d0d0d0', '#606060'])
im = ax.imshow(pop_array, origin='upper', extent=extent, cmap=cmap_grey, vmin=0.5, vmax=2.5, alpha=0.65)
cbar = plt.colorbar(im, ax=ax, shrink=0.6)
cbar.set_ticks([1, 2])
cbar.set_ticklabels(['1 person/cell', '2 persons/cell'])

# Viridis purples for hazards
viridis = plt.cm.viridis
hazard_colors = {
    'point_near': mcolors.to_hex(viridis(0.10)),
    'point_far':  mcolors.to_hex(viridis(0.20)),
    'line':       mcolors.to_hex(viridis(0.30)),
    'polygon':    mcolors.to_hex(viridis(0.40)),
}

# Plot hazards and their buffers
for _, row in hazards.dropna(subset=['geometry']).iterrows():
    g, hid = row.geometry, row.ID_hazard
    col = hazard_colors[hid]

    buf_1 = g.buffer(row.buffer_dist_1 / 111320)
    buf_2 = g.buffer(row.buffer_dist_2 / 111320)

    ax.fill(*buf_2.exterior.xy, alpha=0.1, color=col, zorder=3)
    ax.plot(*buf_2.exterior.xy, color=col, lw=0.8, ls='--', zorder=3)
    ax.fill(*buf_1.exterior.xy, alpha=0.15, color=col, zorder=4)
    ax.plot(*buf_1.exterior.xy, color=col, lw=0.8, zorder=4)

    if g.geom_type == 'Point':
        ax.plot(g.x, g.y, 'o', ms=1, color=col, zorder=5)
    elif g.geom_type == 'LineString':
        ax.plot(*g.xy, lw=3, color=col, zorder=5)
    elif g.geom_type == 'Polygon':
        x, y = g.exterior.xy
        ax.fill(x, y, alpha=0.5, color=col, zorder=5)

# Cell gridlines
tick_positions = np.arange(0, N_CELLS * cell_deg + cell_deg, cell_deg)
ax.set_xticks(tick_positions, minor=False)
ax.set_yticks(tick_positions, minor=False)
ax.grid(True, which='major', color='black', linewidth=0.3, alpha=0.4, zorder=6)

# Hide dense tick labels, show every 10th
n = 10
ax.set_xticklabels([f'{v:.2f}' if i % n == 0 else '' for i, v in enumerate(tick_positions)])
ax.set_yticklabels([f'{v:.2f}' if i % n == 0 else '' for i, v in enumerate(tick_positions)])

ax.set_xlabel('Longitude (deg)')
ax.set_ylabel('Latitude (deg)')
ax.set_title('Hazards and hazard buffer zones from test case\n overlayed on test raster')
plt.tight_layout()
plt.show()

## Expected exposed population counts

The raster has **1 person per km²** in the bottom half (y = 0–50 km) and **2 persons per km²** in the top half (y = 50–100 km).

### point_near — (30, 30) km, bottom half, 1 person/km²
- **2 km buffer**: circle area = π × 2² ≈ 13 km² → **~13 people**
- **5 km buffer**: circle area = π × 5² ≈ 79 km² → **~79 people** 

### point_far — (70, 70) km, top half, 2 persons/km²
- **2 km buffer**: circle area = π × 2² ≈ 13 km² × 2 → **~25 people**
- **5 km buffer**: circle area = π × 5² ≈ 79 km² × 2 → **~157 people**

### line — (10, 10) to (10, 20) km, bottom half, 1 person/km²
- **2 km buffer**: rectangle 10 × 4 km + two semicircles = 40 + π × 2² ≈ 53 km² → **~53 people**
- **5 km buffer**: rectangle 10 × 10 km + two semicircles = 100 + π × 5² ≈ 179 km² → **~179 people** 

### polygon — 4×4 km square centred at (60, 60) km, top half, 2 persons/km²
- **2 km buffer**: square interior 16 km² + buffer ring ~45 km² = ~61 km² × 2 → **~120 people**
- **5 km buffer**: square interior 16 km² + buffer ring ~124 km² = ~140 km² × 2 → **~280 people** 

### missing_geom
- No geometry — dropped by popexposure with a warning, no row returned.

In [ ]:
## Expected exposed population counts


expected = pd.DataFrame([
    {"geometry_id": "point_near", "exposed_2km": 12,  "exposed_5km": 78},
    {"geometry_id": "point_far", "exposed_2km": 25,  "exposed_5km": 157},
    {"geometry_id": "line", "exposed_2km": 52,  "exposed_5km": 178},
    {"geometry_id": "polygon", "exposed_2km": 121, "exposed_5km": 350},
])

print(expected)


In [ ]:
# Calculate things with popexposure

# Initialise the estimator with just the population raster (no admin units)
estimator = popexposure.PopEstimator(pop_data=raster_path)

# Hazard-specific exposure: one row per hazard, columns exposed_1 and exposed_2
results_df = estimator.est_exposed_pop(
    hazard_data=hazards_path,
    hazard_specific=True,
)

print(results_df)
